# Example notebook for Ensemble of latitude profiles and seasonal cycles


##### This notebook shows how the ensemble module can be used for 1D data: the seasonal cycles of the Timeseries diagnostic and the latitude profiles of the LatLon Profiles diagnostic.


In [ ]:
import xarray as xr

from aqua.diagnostics import EnsembleProfiles, PlotEnsembleProfiles, merge_from_data_files

##### Location of the output of each realization


In [ ]:
# Folder with the output of `aqua analysis` for each realization: <outputdir>/<catalog>/<model>/<exp>
exp_dir = "/path/to/outputdir/climatedt-gen2/IFS-NEMO-10km/baseline-hist"
prefix = "climatedt-gen2.IFS-NEMO-10km.baseline-hist"
realizations = ["r2", "r3", "r4"]
names = {
    "catalog_list": "climatedt-gen2",
    "model_list": "IFS-NEMO-10km",
    "exp_list": "baseline-hist",
    "source_list": "lra-r100-monthly",
}
outputdir = "./output"

##### Seasonal cycle
##### The files of the realizations are merged along the `ensemble` dimension.


In [ ]:
file_list = [
    f"{exp_dir}/{r}/atmosphere2d/netcdf/atmosphere2d.seasonalcycles.{prefix}.{r}.2t.monthly.global.nc" for r in realizations
]
cycle = merge_from_data_files(variable="2t", data_path_list=file_list)
cycle

##### A different `diagnostic_product` for each kind of profile keeps the output files apart.


In [ ]:
ens_cycle = EnsembleProfiles(
    var="2t",
    dataset=cycle,
    diagnostic_product="EnsembleSeasonalCycles",
    region="global",
    outputdir=outputdir,
    **names,
)
ens_cycle.run()

##### The reference is read from the output of one realization.


In [ ]:
ref_file = (
    f"{exp_dir}/r2/atmosphere2d/netcdf/atmosphere2d.seasonalcycles.obs.BERKELEY-EARTH.aqua-filled.r1.2t.monthly.global.nc"
)
ref_cycle = xr.open_dataset(ref_file)["2t"]

plot_cycle = PlotEnsembleProfiles(
    diagnostic_product="EnsembleSeasonalCycles",
    region="global",
    ref_catalog="obs",
    ref_model="BERKELEY-EARTH",
    ref_exp="aqua-filled",
    outputdir=outputdir,
    **names,
)
fig = plot_cycle.plot(
    var="2t",
    mean=ens_cycle.dataset_mean,
    std=ens_cycle.dataset_std,
    members=cycle["2t"],
    ref=ref_cycle,
    coord="month",
    data_name="seasonal_cycle",
    save_format="png",
)

##### Latitude profiles
##### Each period has its own file, so the profiles are stacked along a `period` dimension.


In [ ]:
seasons = ["DJF", "MAM", "JJA", "SON"]
periods = {"longterm": "longterm", **{season: f"seasonal.{season.lower()}" for season in seasons}}

profiles = []
for key in periods.values():
    file_list = [
        f"{exp_dir}/{r}/atmosphere2d/netcdf/atmosphere2d.zonal_profile.{prefix}.{r}.{key}.2t.nc" for r in realizations
    ]
    profiles.append(merge_from_data_files(variable="2t", data_path_list=file_list).drop_vars("season", errors="ignore"))

profile = xr.concat(profiles, dim=xr.DataArray(list(periods), dims="period", name="period"))
profile

In [ ]:
ens_profile = EnsembleProfiles(var="2t", dataset=profile, outputdir=outputdir, **names)
ens_profile.run()

##### One panel per season


In [ ]:
plot_profile = PlotEnsembleProfiles(outputdir=outputdir, **names)
fig = plot_profile.plot(
    var="2t",
    mean=ens_profile.dataset_mean,
    std=ens_profile.dataset_std,
    members=profile["2t"],
    panels={season: {"period": season} for season in seasons},
    data_name="seasonal",
    figure_size=(12, 8),
    save_format="png",
)